# RAG (file_search) test

**QE Perspective:** We validate end-to-end RAG: create a vector store, upload a document, ingest it, then call the Responses API with `file_search` and assert the answer is grounded in the document. We also test vector store search, file processing status polling, annotation extraction, and comparing answers with vs without RAG context.

- **Positive:** Vector store + file upload + file_search -> response contains expected fact.
- **Search:** Direct vector store search returns relevant chunks.
- **Annotations:** RAG response includes file citation annotations.
- **Comparison:** Answer with vector store differs from answer without (proves RAG is used).
- **Negative:** Request with non-existent vector_store_id -> API returns error.
- **Edge:** Assert base_url/model set; assert at least one vector_io provider before running.

Config: `BASE_URL`, `INFERENCE_MODEL` (optional: `EMBEDDING_MODEL`, `EMBEDDING_DIMENSION`). Run via pytest or interactively.

## Setup

Load config from env; create client and ensure vector_io provider exists.

In [ ]:
import os
from openai import OpenAI


base_url = os.environ.get("BASE_URL")
model = os.environ.get("INFERENCE_MODEL")
embedding_model = os.environ.get("EMBEDDING_MODEL")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION"))

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

openai_base_url = base_url.rstrip("/")
openai_base_url = (
    openai_base_url if openai_base_url.endswith("/v1") else openai_base_url + "/v1"
)
client = OpenAI(api_key="no-key-needed", base_url=openai_base_url)

providers_resp = client.get("/providers", cast_to=object)
vector_providers = [
    p for p in providers_resp.get("data", []) if p.get("api") == "vector_io"
]
assert vector_providers, "No vector_io provider available"
vector_provider_id = vector_providers[0]["provider_id"]
print(f"Using model: {model}")
print(f"Using embedding model: {embedding_model}")
print(f"Using vector_io provider: {vector_provider_id}")

## Positive: Create vector store, upload doc, ingest, and query

Full RAG pipeline with a known document. Assert the answer is grounded in the document content.

In [ ]:
from io import BytesIO
from uuid import uuid4

doc_text = (
    "Bering Land Bridge National Preserve. "
    "Proclaimed a national monument Dec. 1, 1978; "
    "established as a national preserve Dec. 2, 1980.\n"
    "Denali National Park. Established as Mt. McKinley National Park "
    "Feb. 26, 1917. Designated Denali National Park and Preserve Dec. 2, 1980."
)
question = "When was Bering Land Bridge established as a national preserve?"

vector_store = None
uploaded_file = None
vector_store = client.vector_stores.create(
    name=f"rag_test_{uuid4().hex[:8]}",
    extra_body={
        "provider_id": vector_provider_id,
        "embedding_model": embedding_model,
    },
)
assert vector_store.id, "Vector store creation returned no ID"
print(f"Created vector store: {vector_store.id}")

file_buffer = BytesIO(doc_text.encode("utf-8"))
file_buffer.name = "rag_doc.txt"
uploaded_file = client.files.create(file=file_buffer, purpose="assistants")
assert uploaded_file.id, "File upload returned no ID"
print(f"Uploaded file: {uploaded_file.id}")

# create_and_poll blocks until indexing reaches a terminal state, so
# file_search below never races an in-progress file.
vs_file = client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id,
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 256, "chunk_overlap_tokens": 32},
    },
)
assert vs_file.status == "completed", (
    f"File indexing did not complete (status: {vs_file.status}, "
    f"error: {getattr(vs_file, 'last_error', None)})"
)
print(f"File indexed, status: {vs_file.status}")

response = client.responses.create(
    model=model,
    instructions="Use file_search to answer the question using the provided documents.",
    input=[{"role": "user", "content": question}],
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
    tool_choice={"type": "file_search"},
    include=["file_search_call.results"],
    stream=False,
)
assert response.status == "completed", (
    f"Expected status completed, got {response.status}"
)
text = response.output_text
assert text, "Expected non-empty response text"
assert "1980" in text or "1978" in text, (
    f"Expected year from doc in answer, got: {text[:300]}"
)
print(f"RAG answer: {text[:200]}")

# Prove file_search actually retrieved chunks (via include=file_search_call.results),
# not just that the answer happened to contain the fact.
fs_calls = [item for item in response.output if item.type == "file_search_call"]
assert fs_calls, "Expected a file_search_call item in response output"
fs_results = fs_calls[0].results or []
assert fs_results, "file_search returned no retrieved chunks"
retrieved = " ".join(r.text for r in fs_results)
assert "Bering" in retrieved or "1980" in retrieved or "1978" in retrieved, (
    f"Retrieved chunks missing expected content, got: {retrieved[:300]}"
)
print(f"file_search retrieved {len(fs_results)} chunk(s) with relevant content")

## Vector store search

Query the vector store directly to verify document chunks were indexed and are searchable.

In [7]:
search_response = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query="Bering Land Bridge",
    max_num_results=3,
)
assert hasattr(search_response, "data"), "Search response missing data field"
assert len(search_response.data) > 0, "Search returned no results"
first_result = search_response.data[0]
assert hasattr(first_result, "content"), "Search result missing content"
print(f"Search returned {len(search_response.data)} result(s)")
for i, r in enumerate(search_response.data, 1):
    snippet = ""
    for c in r.content:
        if hasattr(c, "text"):
            snippet = c.text[:100]
            break
    print(f"  Result {i}: score={getattr(r, 'score', 'N/A')}, text={snippet}...")

Search returned 1 result(s)
  Result 1: score=2.0316883133116255, text=Bering Land Bridge National Preserve. Proclaimed a national monument Dec. 1, 1978; established as a ...


## Annotations: verify file citations in RAG response

When using file_search, the response should include file citation annotations referencing the source document.

In [8]:
ann_response = client.responses.create(
    model=model,
    instructions="Answer using the provided documents. Cite your sources.",
    input=[
        {
            "role": "user",
            "content": "When was Denali designated a national park and preserve?",
        }
    ],
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
    tool_choice={"type": "file_search"},
    stream=False,
)
assert ann_response.status == "completed"
has_annotations = False
for output_item in ann_response.output:
    if hasattr(output_item, "content") and output_item.content:
        for content_block in output_item.content:
            if hasattr(content_block, "annotations") and content_block.annotations:
                has_annotations = True
                citations = [
                    a
                    for a in content_block.annotations
                    if getattr(a, "type", None) == "file_citation"
                ]
                print(f"Found {len(citations)} file citation(s)")
                for c in citations:
                    print(f"  file: {getattr(c, 'filename', 'N/A')}")
if not has_annotations:
    print("WARNING: No annotations found in response (server may not support them yet)")

## Comparison: answer with vs without vector store

Verify that RAG actually changes the answer. Without the vector store, the model should not know the specific fact.

In [ ]:
comparison_q = "When was Bering Land Bridge proclaimed a national monument?"

resp_without = client.responses.create(
    model=model,
    input=comparison_q,
    stream=False,
)
text_without = resp_without.output_text

resp_with = client.responses.create(
    model=model,
    instructions="Answer using only the provided documents.",
    input=[{"role": "user", "content": comparison_q}],
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
    tool_choice={"type": "file_search"},
    stream=False,
)
text_with = resp_with.output_text

assert resp_with.status == "completed"
assert "1978" in text_with, f"Expected '1978' in RAG answer, got: {text_with[:300]}"
print(f"Without RAG: {text_without[:150]}")
print(f"With RAG:    {text_with[:150]}")

## Negative: invalid vector_store_id

Using a non-existent vector store ID should raise an error.

In [10]:
error_raised = False
non_grounded = False
try:
    r = client.responses.create(
        model=model,
        input=[{"role": "user", "content": "What is 2+2?"}],
        tools=[{"type": "file_search", "vector_store_ids": ["vs_nonexistent_invalid"]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
    # Check that response contains no file_search grounding
    has_citations = False
    for output_item in r.output:
        if hasattr(output_item, "content"):
            for content_block in output_item.content:
                if hasattr(content_block, "annotations") and content_block.annotations:
                    for ann in content_block.annotations:
                        if getattr(ann, "type", None) == "file_citation":
                            has_citations = True
    non_grounded = not has_citations
except Exception:
    error_raised = True
assert error_raised or non_grounded, (
    "Invalid vector_store_id should either raise an error or return without file citations"
)
print(f"Invalid vector store: error={error_raised}, non_grounded={non_grounded}")

Invalid vector store: error=False, non_grounded=True


## Cleanup

Remove the vector store and uploaded file created during the test.

In [ ]:
# Cleanup: delete vector store and file
if "vector_store" in globals() and vector_store:
    try:
        client.vector_stores.delete(vector_store_id=vector_store.id)
        print(f"Deleted vector store: {vector_store.id}")
    except Exception as e:
        print(f"Warning: failed to delete vector store: {e}")
if "uploaded_file" in globals() and uploaded_file:
    try:
        client.files.delete(file_id=uploaded_file.id)
        print(f"Deleted file: {uploaded_file.id}")
    except Exception as e:
        print(f"Warning: failed to delete file: {e}")